# Graph Models - Cross-Validation

The following tutorial will demo how to run Cross-Validation scripts for the graph models. These methods condense parameters needed when generating graph models as demoed in [graph_model.ipynb]("graph_model.ipynb").

## Import Modules
Import the modules needed to build the model and run cross-validation.

In [1]:
import pandas as pd
from mminer_graphs.model import GCNNet
from mminer_graphs.model import CrossVal

## Dataset Prep
This tutorial will use the bace dataset taken from [MoleculeNet](https://moleculenet.org). This dataset contains additional columns with train/test split labels that were randomly assigned. The labels for these columns are CVx where x is the fold number. The relavent information needed for this model will be extracted as follows:

In [2]:
data = pd.read_csv('dataset/bace.csv')
cols = ['CID', 'smiles', 'Class', 'CV0', 'CV1', 'CV2', 'CV3', 'CV4']
dataset = data[cols]
dataset.head()

,CID,smiles,Class,CV0,CV1,CV2,CV3,CV4
0,BACE_1,O1CC[C@@H](NC(=O)[C@@H](Cc2cc3cc(ccc3nc2N)-c2c...,1,train,train,train,train,train
1,BACE_2,Fc1cc(cc(F)c1)C[C@H](NC(=O)[C@@H](N1CC[C@](NC(...,1,train,train,train,train,train
2,BACE_3,S1(=O)(=O)N(c2cc(cc3c2n(cc3CC)CC1)C(=O)N[C@H](...,1,train,train,train,train,test
3,BACE_4,S1(=O)(=O)C[C@@H](Cc2cc(O[C@H](COCC)C(F)(F)F)c...,1,train,train,train,train,test
4,BACE_5,S1(=O)(=O)N(c2cc(cc3c2n(cc3CC)CC1)C(=O)N[C@H](...,1,train,train,train,train,train


### Optional
As mentioned in the [graph_model.ipynb]("graph_model.ipynb") tutorial, additional features can be added to the model. An example of generating this is as follows:

In [3]:
# extra features
extra_cols = ["pIC50", "MW", "AlogP", "HBA", "HBD", "RB", "HeavyAtomCount", "ChiralCenterCount",
              "ChiralCenterCountAllPossible", "RingCount"]
extra_data = data[extra_cols]
extra_list = extra_data.values.tolist()

## Build Model
As of this writing, the mol-miner-graphs contains two models. Both share similar parameters, with the exception of GINENet, which also considers graph edge_features. In this example, the basic GCNNet will be used.

In [4]:
model = GCNNet(node_dim=142, hidden_dim=64, graph_layers=2, mlp_layers=2, reduce_graph=False, mlp_dropout=0, num_classes=2, add_feats=extra_list)

In [5]:
model

GCNNet(
  (gcn_conv): ModuleList(
    (0): GCNConv(142, 64)
    (1): GCNConv(64, 64)
  )
  (mlp): ModuleList(
    (0): Sequential(
      (0): Linear(in_features=74, out_features=37, bias=True)
      (1): BatchNorm1d(37, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout(p=0, inplace=False)
    )
  )
  (final_mlp): Linear(in_features=37, out_features=2, bias=True)
)

## Cross-Validation
Instantiating the **CrossVal()** class can accept three parameters. The model and dataset parameters are optional and if are not used when instantiating **CrossVal()**, then will need to be used under the method **get_cv_scores()**. Here, the model and dataset is given to the **CrossVal()** class. Additionally, there is an **estop** parameter. This is the early stopping parameter and requires an int that will determine the number of epochs a given CV fold will run before stopping. Additionally, users can use the mol-miner-graphs **EarlyStopping()** class to create a custom monitor and mode to trigger early stopping.

In [6]:
cv = CrossVal(model=model, dataset=dataset, estop=100)

## Get CV Scores
The **get_cv_scores()** method contains many parameters and should mirror what is needed to run the models individually as seen in [graph_model.ipynb]("graph_model.ipynb"). However, it comes with additional parameters unique for its function. The pertinent parameters are shown in this tutorial.

By default, this method will convert the smiles strings from the input dataset using **AtomGraph**. Additional features, if needed, must be given to this method as a list. Finally a progress bar and verbosity for each CV Round can be Turned on or off depending on user needs.

In [7]:
metrics = cv.get_cv_scores(graph_type="AtomGraph", labels="CV", cv=5, smi_col='smiles', activity_col='Class', device='cpu', epochs=350, batch_size=32, add_feats=extra_list, verbose=False, pbar=True)

CV Round:   0%|          | 0/5 [00:00<?, ?it/s]

Model and Tensor running on cpu


CV Round:  20%|██        | 1/5 [01:06<04:27, 66.89s/it]

Early stopping triggered after 195 epochs!
Average Training Loss: 0.155
Average Test Loss: 1.065
Average Training Accuracy: 0.937
Average Test Accuracy: 0.767
Model and Tensor running on cpu


CV Round:  40%|████      | 2/5 [01:49<02:38, 52.76s/it]

Early stopping triggered after 114 epochs!
Average Training Loss: 0.161
Average Test Loss: 1.159
Average Training Accuracy: 0.934
Average Test Accuracy: 0.744
Model and Tensor running on cpu


... (2 outputs left out)

CV Round:  80%|████████  | 4/5 [02:58<00:40, 40.36s/it]

Early stopping triggered after 100 epochs!
Average Training Loss: 0.166
Average Test Loss: 1.249
Average Training Accuracy: 0.932
Average Test Accuracy: 0.731
Model and Tensor running on cpu


CV Round: 100%|██████████| 5/5 [03:35<00:00, 43.10s/it]

Early stopping triggered after 106 epochs!
Average Training Loss: 0.168
Average Test Loss: 1.191
Average Training Accuracy: 0.931
Average Test Accuracy: 0.738


## CV Metrics
The output is given as a pd.DataFrame. This can be saved as a .csv file or, depending on the user needs, modified for further analysis.

In [8]:
metrics

,Accuracy,Precision,Recall,F1,MCC,ROC_AUC,Accuracy_ave,Precision_ave,Recall_avg,F1 Score_avg,MCC_avg,ROC_AUC_avg,Loss,Loss_ave
0,"[0.767, 0.744, 0.735, 0.731, 0.738]","[0.751, 0.705, 0.684, 0.677, 0.688]","[0.904, 0.915, 0.916, 0.913, 0.917]","[0.793, 0.768, 0.755, 0.752, 0.762]","[0.575, 0.552, 0.542, 0.534, 0.544]","[0.758, 0.75, 0.747, 0.743, 0.748]",0.743,0.701,0.913,0.766,0.549,0.749,"[1.065, 1.159, 1.191, 1.249, 1.191]",1.171
